# Анализ ошибок

## Импоот библиотек

In [4]:
import pandas as pd 
import numpy as np

## Загрузка данных

In [5]:
import json
with open("../outputs/best_results_cheap_gen.json", "r") as f:
    data = json.load(f)

errors_df = pd.DataFrame(data["predictions"])

## Общий анализ

In [6]:
EX = errors_df["execution_match"].sum() / errors_df["execution_match"].count() * 100
print("Процент корреткного EX:", EX)
print("Процент ошибок:", 100 - EX)

ex_errors_count = errors_df.query("execution_match == False").shape[0]
print("Всего ошибок EX:", ex_errors_count)

Процент корреткного EX: 74.95164410058027
Процент ошибок: 25.048355899419732
Всего ошибок EX: 259


## Самые проблемые БД по EX

In [7]:
print("Самые проблемые БД:")
display(errors_df.query("execution_match == False").groupby("db_id")["execution_match"].count().sort_values(ascending=False)) 

print()
print("Самые проблемые БД (% от всех ошибок):")
display(errors_df.query("execution_match == False").groupby("db_id")["execution_match"].count().sort_values(ascending=False) / ex_errors_count * 100)

Самые проблемые БД:


db_id
car_1                           51
student_transcripts_tracking    43
world_1                         25
dog_kennels                     23
flight_2                        22
employee_hire_evaluation        17
concert_singer                  12
cre_Doc_Template_Mgt            10
tvshow                          10
network_1                        9
wta_1                            9
orchestra                        8
poker_player                     5
pets_1                           3
museum_visit                     3
battle_death                     3
real_estate_properties           2
voter_1                          2
course_teach                     2
Name: execution_match, dtype: int64


Самые проблемые БД (% от всех ошибок):


db_id
car_1                           19.691120
student_transcripts_tracking    16.602317
world_1                          9.652510
dog_kennels                      8.880309
flight_2                         8.494208
employee_hire_evaluation         6.563707
concert_singer                   4.633205
cre_Doc_Template_Mgt             3.861004
tvshow                           3.861004
network_1                        3.474903
wta_1                            3.474903
orchestra                        3.088803
poker_player                     1.930502
pets_1                           1.158301
museum_visit                     1.158301
battle_death                     1.158301
real_estate_properties           0.772201
voter_1                          0.772201
course_teach                     0.772201
Name: execution_match, dtype: float64

## Жесткие ошибки

### По БД

In [8]:
strict_errors = errors_df[errors_df["error_message"].notna()]

print("Всего жестких ошибок:", strict_errors.shape[0])
print("Самые проблемые БД по жестким ошибкам:")
display(strict_errors.groupby("db_id")["error_message"].count().sort_values(ascending=False))

print()
print("Самые проблемые БД по жестким ошибкам (% от всех ошибок):")
display(strict_errors.groupby("db_id")["error_message"].count().sort_values(ascending=False) / strict_errors.shape[0] * 100)

Всего жестких ошибок: 37
Самые проблемые БД по жестким ошибкам:


db_id
car_1                           19
student_transcripts_tracking     5
concert_singer                   3
employee_hire_evaluation         3
world_1                          3
dog_kennels                      2
wta_1                            2
Name: error_message, dtype: int64


Самые проблемые БД по жестким ошибкам (% от всех ошибок):


db_id
car_1                           51.351351
student_transcripts_tracking    13.513514
concert_singer                   8.108108
employee_hire_evaluation         8.108108
world_1                          8.108108
dog_kennels                      5.405405
wta_1                            5.405405
Name: error_message, dtype: float64

## По запросам

In [14]:
display(strict_errors.groupby("error_message")["error_message"].count().sort_values(ascending=False))

error_message
benchmark_timeout: example exceeded 660s                                                                                                                                                                                                                                                                   6
schema_validation: unknown table or alias 'countrylanguage' for column 'CountryCode'                                                                                                                                                                                                                       3
schema_validation: unknown table 'car'                                                                                                                                                                                                                                                                     3
schema_validation: unknown column 'T2.Maker'                                       

## Warnings

In [18]:
errors_df["warnings"]

0       [selector: padded reranker selection with vect...
1       [selector: padded reranker selection with vect...
2       [selector: padded reranker selection with vect...
3       [selector: padded reranker selection with vect...
4       [selector: padded reranker selection with vect...
                              ...                        
1029    [selector: padded reranker selection with vect...
1030               [value_linker: found 3 column hint(s)]
1031    [value_linker: found 1 value hint(s), value_li...
1032    [selector: padded reranker selection with vect...
1033    [value_linker: found 2 column hint(s), query_s...
Name: warnings, Length: 1034, dtype: object

In [19]:
errors_df["warnings"] = errors_df["warnings"].astype(str)
errors_df.groupby("warnings")["warnings"].count().sort_values(ascending=False)

warnings
['selector: padded reranker selection with vector candidates', 'value_linker: found 3 column hint(s)']                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [39]:
for w in sorted(errors_df["warnings"].unique()):
    print(w)


["execution_filter: (sqlite3.OperationalError) no such column: edispl\n[SQL: SELECT avg(edispl) FROM car_makers WHERE Maker = 'volvo';]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)", "execution_filter: (sqlite3.OperationalError) no such column: edispl\n[SQL: SELECT avg(edispl) FROM car_makers WHERE Maker = 'volvo';]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)", "execution_filter: (sqlite3.OperationalError) no such column: edispl\n[SQL: SELECT avg(edispl) FROM car_makers WHERE Maker = 'volvo';]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)", "execution_filter: (sqlite3.OperationalError) no such column: edispl\n[SQL: SELECT avg(edispl) FROM car_makers WHERE Maker = 'volvo';]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)", "execution_filter: (sqlite3.OperationalError) no such column: edispl\n[SQL: SELECT avg(edispl) FROM car_makers WHERE Maker = 'volvo';]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)", "exe

## Гипотезы улучшеия

1. **8 БД составляют почти 80% ошибок execution match**: `['car_1', 'student_transcripts_tracking', 'world_1', 'dog_kennels', 'flight_2', 'employee_hire_evaluation', 'concert_singer', 'cre_Doc_Template_Mgt']`
2. **основные жесткие ошибки:** 
    - schema-linker или модель генерации выдумывают колонки, пишут некорретные;
    - timeout exceeded;
    - Could not decode to UTF-8 column;


In [70]:
car1_err = errors_df.query("execution_match == False & db_id == 'car_1'")[["db_id", "question", "predicted_sql", "gold_sql"]]#["question"].iloc[0]

In [80]:
car1_err.iloc[5:7]["question"].tolist()

['Find the make and production time of the cars that were produced in the earliest year?',
 'What is the maker of the carr produced in the earliest year and what year was it?']

## Выпонление проблемных запросов

In [50]:
import sqlite3

In [67]:
connection = sqlite3.connect("/home/vadim/HSE_EDUCATION/text-to-sql-hse26/databases/spider/database/car_1/car_1.sqlite")
cursor = connection.cursor()

# 2. Create a table
cursor.execute("SELECT T3.Model FROM cars_data AS T1 JOIN car_names AS T2 ON T1.Id = T2.MakeId JOIN model_list AS T3 ON T2.Model = T3.Model WHERE T1.Weight < (SELECT avg(Weight) FROM cars_data);")
p = cursor.fetchall()

cursor.execute("SELECT T1.model FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId  =  T2.Id WHERE T2.Weight  <  (SELECT avg(Weight) FROM CARS_DATA);")
g = cursor.fetchall()

In [68]:
len(p)

229

In [69]:
len(g)

230